In [1]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from transformers import XLNetTokenizer
import sys
import os

# add src to path
sys.path.append(os.path.abspath('..'))

from src.dataset import get_datasets
from src.model import DualHeadXLNet

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"device: {device}")

device: cpu


In [2]:
# init tokenizer
tokenizer = XLNetTokenizer.from_pretrained('xlnet-base-cased')

# load real data
train_ds, test_ds = get_datasets(
    '../data/processed/train.csv', 
    '../data/processed/test.csv', 
    tokenizer
)

# pick subset from TRAIN DS
mini_batch = torch.utils.data.Subset(train_ds, range(8))
loader = DataLoader(mini_batch, batch_size=4, shuffle=True)

# inspect one batch
batch = next(iter(loader))
print("input shape:", batch['input_ids'].shape)
print("clarity labels:", batch['clarity_labels'])
print("evasion labels:", batch['evasion_labels'])

spiece.model:   0%|          | 0.00/798k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.38M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/760 [00:00<?, ?B/s]

loading train data...
loading test data (with voting)...
Train size: 3448 | Test size: 308
input shape: torch.Size([4, 512])
clarity labels: tensor([1, 1, 1, 0])
evasion labels: tensor([4, 3, 2, 0])


In [3]:
# init model
model = DualHeadXLNet().to(device)
model.train()

optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4) # high lr for fast overfit
loss_fn = nn.CrossEntropyLoss()

print("starting sanity check...")

for epoch in range(30):
    total_loss = 0
    for batch in loader:
        ids = batch['input_ids'].to(device)
        mask = batch['attention_mask'].to(device)
        c_true = batch['clarity_labels'].to(device)
        e_true = batch['evasion_labels'].to(device)
        
        optimizer.zero_grad()
        
        c_logits, e_logits = model(ids, mask)
        
        loss = loss_fn(c_logits, c_true) + loss_fn(e_logits, e_true)
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
        
    if epoch % 5 == 0:
        print(f"epoch {epoch}: loss {total_loss:.4f}")

print(f"final loss: {total_loss:.4f}")
# should be < 0.1

pytorch_model.bin:   0%|          | 0.00/467M [00:00<?, ?B/s]

starting sanity check...


model.safetensors:   0%|          | 0.00/467M [00:00<?, ?B/s]

epoch 0: loss 10.5601
epoch 5: loss 5.0449
epoch 10: loss 3.7941
epoch 15: loss 2.8327
epoch 20: loss 1.9083
epoch 25: loss 0.0246
final loss: 0.0893
